# Q1 v2: Crash types and road design in the KSI model (draft)

Does adding past red-light-running, angle and rear-end crashes (#26) and LION road features (`preprocessing/roadway.py`) to the Q1 model improve future KSI prediction? (issue #32)

**Changes from Q1**
- **Sites are LION intersections** (#36), not 100 ft crash clusters: nodes where 3+ streets meet, with nodes of one divided-road intersection merged within 60 ft. A crash belongs to the nearest site within 100 ft.
- **Pooled 3-year test** (#31): fit once on prediction year 2022, score 2023–2025, and compare each site's KSI total over those three years.

**Crash types** (`CrashConfig`): *red-light running* is any vehicle with contributing factor "Traffic Control Disregarded" (a proxy that also covers stop signs); *rear-end* is one vehicle hit on its front and another on its back; *angle* is one hit on its front and another on its side.

In [1]:
import sys
import warnings

import numpy as np
import pandas as pd

sys.path.append("..")
warnings.filterwarnings("ignore")

from model.count_models import HistoryRanker, NegativeBinomialModel
from preprocessing.config import SiteConfig
from preprocessing.panel import load_panel
from preprocessing.roadway import ROADWAY_FEATURES, add_roadway_features, load_roadway_features
from validation.validate import pooled_holdout

site_config = SiteConfig(source="lion")
panel, scoring, sites = load_panel(site_config=site_config)
panel = add_roadway_features(panel, load_roadway_features(sites, site_config=site_config))
print(f"{panel['site_id'].nunique():,} intersection sites, prediction years {sorted(panel['year'].unique())}")
panel.groupby("year")[["ksi", "rlr", "rear_end", "angle", "crashes"]].sum()

46,896 intersection sites, prediction years [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,ksi,rlr,rear_end,angle,crashes
year,,,,,
2022,2085,2741,15243,16199,69149
2023,2222,2898,14524,15747,66063
2024,2208,2501,13452,13961,61066
2025,2180,2186,13010,12882,57833


## 1. How common are red-light-running crashes?
Ko et al. (2012) found cameras pay off at intersections with 4+ red-light-running crashes a year. Few NYC intersections reach that by this proxy.

In [2]:
rlr_per_year = panel.query("year == 2025")["rlr_lag3"] / 3
pd.Series({
    "sites with any RLR crash (2022–24)": (rlr_per_year > 0).sum(),
    "1+ per year": (rlr_per_year >= 1).sum(),
    "2+ per year": (rlr_per_year >= 2).sum(),
    "4+ per year (Ko rule)": (rlr_per_year >= 4).sum(),
})

sites with any RLR crash (2022–24)    5903
1+ per year                            450
2+ per year                             30
4+ per year (Ko rule)                    3
dtype: int64

## 2. Model comparison (pooled 2023–2025)
Top-k capture: share of all 2023–25 KSI crashes at the top-k sites (higher is better). Deviance: fit across all sites (lower is better).

In [3]:
Q1 = [f"log1p_{c}_lag3" for c in ["pdo", "injury_nonsevere", "ksi_nonfatal", "fatal", "ped_injured", "cyc_injured"]]
TYPES = [f"log1p_{c}_lag3" for c in ["rlr", "rear_end", "angle"]]
# Every LION site is an intersection, so the leg indicators always sum to 1;
# drop legs_3 to make 3-leg intersections the reference.
ROAD = [f for f in ROADWAY_FEATURES if f != "legs_3"]
models = {
    "Past KSI (baseline)": lambda: HistoryRanker("ksi_lag3", 3),
    "Q1 model": lambda: NegativeBinomialModel(Q1),
    "+ crash types": lambda: NegativeBinomialModel(Q1 + TYPES),
    "+ LION road": lambda: NegativeBinomialModel(Q1 + ROAD),
    "+ both": lambda: NegativeBinomialModel(Q1 + TYPES + ROAD),
}
rows = []
for name, make in models.items():
    metrics, _ = pooled_holdout(make, panel, (2023, 2024, 2025), (100, 1000))
    rows.append(metrics.assign(model=name))
res = pd.concat(rows)
res["column"] = [m if pd.isna(k) else f"{m}{int(k)}" for m, k in zip(res["metric"], res["k"])]
res.pivot(index="model", columns="column", values="value").loc[list(models)].round(4)

column,capture100,capture1000,deviance
model,,,
Past KSI (baseline),0.0236,0.1382,4.9878
Q1 model,0.0312,0.1790,0.4672
+ crash types,0.0322,0.1762,0.4663
+ LION road,0.0301,0.1808,0.4572
+ both,0.0309,0.1805,0.4564


## 3. Rate ratios (full model, trained on 2022)
Crash counts: effect of doubling the past count, holding the others fixed. Road features: effect of one unit (a 0/1 flag, one lane, 10 ft of width, 1 ft of lane width).

In [4]:
train = panel[panel["year"] < 2023]
final = NegativeBinomialModel(Q1 + TYPES + ROAD).fit(train, train["ksi"], groups=train["site_id"])
coef = final.coefficients().loc[TYPES + Q1 + ROAD]
# rate ratios are per unit of log(1 + count); ratio ** log(2) is the effect of doubling the count
logged = coef.index.isin(TYPES + Q1)
coef.loc[logged, ["ratio", "ci_low", "ci_high"]] **= np.log(2)
coef.round(3)

,coef,ratio,ci_low,ci_high,p_value
log1p_rlr_lag3,-0.083,0.944,0.877,1.017,0.129
log1p_rear_end_lag3,-0.061,0.958,0.889,1.033,0.268
log1p_angle_lag3,-0.142,0.906,0.843,0.974,0.008
log1p_pdo_lag3,0.641,1.559,1.441,1.687,0.000
log1p_injury_nonsevere_lag3,0.582,1.497,1.378,1.626,0.000
log1p_ksi_nonfatal_lag3,0.105,1.075,0.990,1.168,0.085
log1p_fatal_lag3,-0.094,0.937,0.751,1.168,0.562
log1p_ped_injured_lag3,-0.001,0.999,0.938,1.064,0.981
log1p_cyc_injured_lag3,0.188,1.139,1.069,1.214,0.000
legs_4,0.213,1.238,1.091,1.403,0.001
